## Setup    

In [1]:
import os, sys
from pathlib import Path

# Finn project root ved å gå opp fra notebooks-mappa
PROJECT_ROOT = Path(__file__).parent.parent if '__file__' in globals() else Path().cwd().parent
assert (PROJECT_ROOT / "src" / "train.py").exists(), f"Finner ikke src/train.py i {PROJECT_ROOT} – sjekk PROJECT_ROOT"

# Gjør at imports og relative paths funker i notebook
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print("OK:", PROJECT_ROOT)

OK: /cluster/home/larshfle/superpoint_transformer


In [2]:
import os
os.environ["WANDB_MODE"] = "disabled"   # eller "offline"
os.environ["WANDB_SILENT"] = "true"


## Create dataset

In [3]:
from pathlib import Path

# Her må du peke til hvor du har norske ALS tiles + labels
#filepath = '/cluster/home/larshfle/datasets/bergen2020_5pkt_train/32-1-468-145-63.laz'
filepath = '/cluster/home/larshfle/datasets/bergen2020_5pkt_train'

NOR_DATA_ROOT = Path(filepath).resolve()

# Bare for å minne deg på å sjekke at det faktisk finnes
print("NOR_DATA_ROOT:", NOR_DATA_ROOT)

NOR_DATA_ROOT: /cluster/home/larshfle/datasets/bergen2020_5pkt_train


In [4]:
import laspy, numpy as np
from src.datasets.norway_config import ID2TRAINID

las = laspy.read(f"{filepath}/32-1-468-145-63.laz")
mapped = ID2TRAINID[las["classification"]]

print("train labels:", np.unique(mapped, return_counts=True))

train labels: (array([0, 1]), array([1808189, 4514801]))


# Model

In [ ]:
import subprocess, sys, os
from pathlib import Path

# Litt mer debug hvis noe feiler
os.environ["HYDRA_FULL_ERROR"] = "1"
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"


cmd = [
    sys.executable, "src/train.py",

    # Bruk ferdig "training recipe"
    "experiment=semantic/dales_11g",

    # Bruk din egen datamodule
    "datamodule=semantic/norway",

    # Hvor data ligger
    f"paths.data_dir={NOR_DATA_ROOT}",

    # ---- super liten test ----
    "trainer.max_epochs=2",
    "trainer.limit_train_batches=10",
    "trainer.limit_val_batches=2",
    "trainer.num_sanity_val_steps=0",

    # Progress bar i notebook
    "trainer.enable_progress_bar=true",

    # Ikke W&B nå
    "logger=csv",
]

print(" ".join(cmd))
subprocess.run(cmd, check=True)
